# Selección de columnas para la red bayesiana — ENMT

**Proyecto:** transporte_UNAM 

**Responsable de esta etapa:** Santiago De La Calleja

**Entrada:** `data/processed/enmt_limpio.csv` + `data/processed/diccionario.json` (salidas de `01_limpieza_enmt.ipynb`) 

**Salida:** `data/processed/enmt_bn.csv` (subconjunto crudo para modelar) + `data/processed/mapa_nodos.csv`

---

## Por qué este notebook

El dataset **analítico** (`enmt_analitico.csv`) no sirve para esta etapa: su recorte temático deja fuera la victimización (`p25_*`), la ocupación (`h21_*`) y la calificación por modo (`p1c_*`), que son justo lo que piden los queries. Partimos entonces del **completo** (`enmt_limpio.csv`).

Como los nombres de la base ya están normalizados y no siempre coinciden con el codebook, aquí **no hardcodeamos** la lista final: para cada nodo damos columnas candidatas y el notebook reporta cuáles existen de verdad y con qué cobertura.

Los 4 queries y sus nodos:

1. ¿Las personas cuyo medio de transporte principal es el metro consideran que el transporte público es tanto más seguro como efectivo que el resto de usuarios de transporte público que NO usan mayormente el metro? (Percepción de **seguridad** y **efectividad**).
2. ¿Existe diferencia entre patrones (jefes) y profesionistas en su percepción del costo de los medios de transporte que utilizan? (**Patrones vs. profesionistas** - percepción del **costo**).
3. ¿El nivel de escolaridad de una persona influye en el medio de transporte que más utiliza? (**Escolaridad** → **modo principal**).
4. Entre las personas que usan transporte público, ¿Es distinta la probabilidad de sufrir asalto en transporte publico si gana más o menos que el salario promedio (entre quienes usan transporte público)? (**Asalto en TP** según **ingreso** ≷ promedio).


## Por qué estas variables

Cada nodo que exploramos aquí responde a una necesidad concreta de los cuatro queries. La lógica de selección fue: partir de lo que cada pregunta exige medir y buscar en la encuesta la variable que mejor lo capta, revisando alternativas cuando el concepto no existe de forma directa.

**Query 1 — Metro vs. resto de usuarios de TP: seguridad y efectividad percibidas.**

El grupo se construye desde `modo_principal`, derivado de la batería `p1a_*` (frecuencia de uso por medio), porque es la única fuente que capta el uso real de transporte, no la opinión. La seguridad percibida es directa: `p17_4` (transporte público seguro/inseguro). La "efectividad" no es una variable de la encuesta, así que se aproxima con `p17_1` (eficiente) y `p17_2` (rápido). Exploramos también la batería `p1c_*` (calificación por modo) como una operacionalización más fina, para decidir con datos cuál conviene.

**Query 2 — Patrones vs. profesionistas - percepción del costo.**

El costo percibido es directo: `p17_3` (transporte público barato/caro). La ocupación detallada solo existe en el roster del hogar (`h21_*`), así que se recupera la ocupación del informante enlazándola con `h8` (individuo seleccionado). `cond_act` (trabaja/no) queda como respaldo, aunque es insuficiente para distinguir tipos de ocupación.

**Query 3 — Escolaridad y modo de transporte principal.**

La escolaridad del informante es `escol`. El modo principal vuelve a derivarse de `p1a_*`. Son las dos piezas mínimas para ver si el nivel educativo se asocia con el medio que más se usa.

**Query 4 — Ingreso y riesgo de asalto en TP.**

El desenlace es directo: `p25_1_2` (víctima de asalto en transporte público). El ingreso es `ing_ind` (tramos en salarios mínimos). El universo (usuarios de TP) se define desde `p1a_*`.

**Variables de contexto (`region`, `tam_loc`, `sexo`, `edad_1`).** 

No responden un query por sí solas, pero la oferta de transporte y los patrones de uso, percepción y victimización varían por lugar y perfil demográfico. Se incluyen para poder controlar por ellas al construir las redes. `pondi2` es el factor de expansión, necesario para cualquier estimación a nivel poblacional.

## 1. Configuración y carga

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 170)

# Raíz robusta (igual criterio que 01_limpieza)
BASE = Path.cwd()
if not (BASE / "data").is_dir():
    BASE = BASE.parent
assert (BASE / "data").is_dir(), f"No encuentro data/ desde {Path.cwd()}"

DIR_PROC = BASE / "data" / "processed"
RUTA_LIMPIO = DIR_PROC / "enmt_limpio.csv"
RUTA_DICC = DIR_PROC / "diccionario.json"

faltan = [p.name for p in (RUTA_LIMPIO, RUTA_DICC) if not p.exists()]
assert not faltan, (
    f"No existen {faltan} en {DIR_PROC}. "
    "data/processed/ está en .gitignore: corre 01_limpieza_enmt.ipynb para regenerarlos."
)

df = pd.read_csv(RUTA_LIMPIO, low_memory=False)
DICC = json.loads(RUTA_DICC.read_text(encoding="utf-8"))["variables"]

def etiqueta(col):
    return DICC.get(col, {}).get("etiqueta", "")

print(f"Base completa: {df.shape[0]:,} filas x {df.shape[1]:,} columnas")
print(f"Diccionario  : {len(DICC):,} variables")

Base completa: 1,191 filas x 556 columnas
Diccionario  : 556 variables


## 2. Mapa de nodos → columnas candidatas

Para cada nodo del modelo se anotan las columnas donde podría estar, en orden de preferencia. Esta sección solo define la lista objetivo. Se listan varias candidatas por nodo porque los nombres del codebook y de la base limpia no siempre coinciden, y aquí evitamos suponer cuál existe.

In [2]:
# nodo, query, tipo, candidatas (en orden de preferencia), nota
MAPA = [
    # --- contexto / confusores ---
    ("region",        "conf", "exact",  ["region"], "región geográfica"),
    ("tam_loc",       "conf", "exact",  ["tam_loc"], "tamaño de localidad"),
    ("sexo",          "conf", "exact",  ["sexo", "sd1"], "sexo del informante"),
    ("edad",          "conf", "exact",  ["edad_1", "sd2"], "edad (numérica -> agrupar)"),
    ("ponderador",    "todos","exact",  ["pondi2", "pondi", "pondi_v", "pondi_h"], "factor de expansión"),
    # --- socioeconómicas ---
    ("escolaridad",   "Q3",   "exact",  ["escol", "sd4"], "driver de Q3"),
    ("ingreso_ind",   "Q4",   "exact",  ["ing_ind", "sd13"], "ingreso individual (numérico)"),
    ("ingreso_fam",   "Q4",   "exact",  ["ing_fam", "sd15"], "respaldo de ingreso"),
    ("cond_act",      "Q2",   "exact",  ["cond_act"], "trabaja/no (fallback de ocupación)"),
    ("ocupacion",     "Q2",   "patron", [r"^h21_\d+$"], "roster: ocupación por integrante"),
    # --- uso de transporte (modo principal / universo TP) ---
    ("uso_modos",     "Q1,Q3,Q4", "patron", [r"^p1a_\d+$"], "frecuencia de uso por modo"),
    # --- percepción (esquema real p17_*; alterno per-modo p1c_*) ---
    ("perc_seguridad","Q1",   "exact",  ["p17_4"], "TP seguro/inseguro"),
    ("perc_eficiente","Q1",   "exact",  ["p17_1"], "TP eficiente/ineficiente"),
    ("perc_rapido",   "Q1",   "exact",  ["p17_2"], "TP rápido/lento"),
    ("perc_costo",    "Q2",   "exact",  ["p17_3"], "TP barato/caro"),
    ("perc_seg_modo", "Q1-alt","patron",[r"^p1c_\d+_2$"], "alterno: seguridad por modo"),
    ("perc_costo_modo","Q2-alt","patron",[r"^p1c_\d+_6$"], "alterno: costo por modo"),
    # --- victimización ---
    ("asalto_tp",     "Q4",   "exact",  ["p25_1_2"], "víctima de asalto en TP"),
]
print(f"Nodos a resolver: {len(MAPA)}")

Nodos a resolver: 18


## 3. Resolución y reporte de cobertura contra la base real

Toma la lista de la sección anterior y la contrasta con la base: confirma qué columnas existen, con qué porcentaje de faltantes, cuántas categorías tienen y qué significan. Es el diagnóstico central del notebook — el que determina si cada query se sostiene con los datos disponibles.

In [3]:
def cobertura(col):
    s = df[col]
    return {
        "col": col,
        "pct_nulos": round(float(s.isna().mean()) * 100, 1),
        "n_unicos": int(s.nunique(dropna=True)),
        "tipo": str(s.dtype),
        "etiqueta": etiqueta(col)[:70],
    }

filas, faltantes = [], []
for nodo, query, tipo, cands, nota in MAPA:
    if tipo == "exact":
        elegida = next((c for c in cands if c in df.columns), None)
        cols = [elegida] if elegida else []
    else:  # patron
        cols = sorted(c for c in df.columns for p in cands if re.fullmatch(p, c))
    if not cols:
        faltantes.append((nodo, query, cands))
        filas.append({"nodo": nodo, "query": query, "estado": "FALTA",
                      "col": " | ".join(cands), "pct_nulos": None,
                      "n_unicos": None, "tipo": None, "etiqueta": nota})
        continue
    for c in cols:
        r = cobertura(c)
        r.update({"nodo": nodo, "query": query, "estado": "ok"})
        filas.append(r)

reporte = pd.DataFrame(filas)[
    ["nodo", "query", "estado", "col", "pct_nulos", "n_unicos", "tipo", "etiqueta"]
]
# Para baterías, resumimos (cuántas columnas y cobertura media) además del detalle
resumen_bat = (reporte[reporte["nodo"].isin(["uso_modos","ocupacion","perc_seg_modo","perc_costo_modo"])]
               .groupby("nodo")
               .agg(n_cols=("col","size"), pct_nulos_medio=("pct_nulos","mean"))
               .round(1))

print("── Nodos de un solo ítem ──")
display(reporte[~reporte["nodo"].isin(resumen_bat.index)].reset_index(drop=True))
print("\n── Baterías (resumen) ──")
display(resumen_bat)

if faltantes:
    print("\n⚠ NODOS NO ENCONTRADOS EN LA BASE (revisar nombre o si están en el crudo):")
    for nodo, query, cands in faltantes:
        print(f"   · {nodo} [{query}] — buscaba: {cands}")

── Nodos de un solo ítem ──


,nodo,query,estado,col,pct_nulos,n_unicos,tipo,etiqueta
0,region,conf,ok,region,0.0,4,int64,Región
1,tam_loc,conf,ok,tam_loc,0.0,4,int64,Tamaño de localidad
2,sexo,conf,ok,sexo,0.0,2,int64,Sexo
3,edad,conf,ok,edad_1,0.0,6,int64,Edad
4,ponderador,todos,ok,pondi2,0.0,933,int64,Factor de expansión individual
5,escolaridad,Q3,ok,escol,0.1,5,float64,Escolaridad
6,ingreso_ind,Q4,ok,ing_ind,0.1,5,float64,Ingreso Individual
7,ingreso_fam,Q4,ok,ing_fam,0.0,8,int64,Ingreso familiar
8,cond_act,Q2,ok,cond_act,0.6,2,float64,Condición de actividad
9,perc_seguridad,Q1,ok,p17_4,0.0,2,int64,"17 Usted, ¿cómo considera el transporte públic..."



── Baterías (resumen) ──


,n_cols,pct_nulos_medio
nodo,,
ocupacion,6,76.6
perc_costo_modo,11,90.7
perc_seg_modo,21,93.5
uso_modos,22,0.8


In [4]:
import re
import json

# 1) Reconstruir la ocupación del informante: h21_{h8}
def ocup_sel(row):
    k = row["h8"]
    if pd.isna(k):
        return pd.NA
    col = f"h21_{int(k)}"
    return row[col] if col in df.columns else pd.NA

df["ocupacion_ind"] = df.apply(ocup_sel, axis=1)

# 2) Etiquetas de ocupación (h21 comparte catálogo)
etiquetas_ocup = json.loads(
    (DIR_PROC / "diccionario.json").read_text(encoding="utf-8")
)["variables"]["h21_1"]["valores"]

# 3) Tabla con nombres y conteos
tabla = (
    df["ocupacion_ind"]
    .value_counts(dropna=False)
    .rename(index=lambda k: etiquetas_ocup.get(str(int(k)), "Sin dato")
            if pd.notna(k) else "NaN")
    .rename_axis("ocupacion")
    .reset_index(name="n")
)
print(tabla.to_string(index=False))

                                                  ocupacion   n
                                                        NaN 489
                                                Comerciante 121
                               Trabajador por cuenta propia  72
   Trabajador en actividades  de reparacion y mantenimiento  69
                                              Profesionista  62
                 Trabajador en actividades  administrativas  58
                                                    Tecnico  55
                         Trabajador en actividad industrial  51
                   Empleado de comercio y agente  de ventas  46
                               Otro trabajador en servicios  45
Trabajador en actividades  agricolas, ganaderas, silvicolas  39
    Vendedor ambulante y  trabajador ambulante en servicios  24
                                 Trabajador de la educacion  23
                         Trabajador en servicios domesticos  22
                              Ocupacione

## 4. Punto a abordar: Dificultad estadística para el Q2

**Planteamiento original:** comparar la percepción del costo del transporte público entre
**patrones** y **profesionistas**.

**Problema encontrado:** al recuperar la ocupación del informante, solo 3 personas se clasifican como "Patrón". Con esa cantidad no es posible estimar de forma confiable las probabilidades asociadas a esa categoría: cualquier cruce con la percepción de costo produce celdas de 0 a 2 casos, cuyas probabilidades no tienen sustento estadístico.

**Ajuste propuesto:** conservar la idea principal de la pregunta (contrastar dos posiciones ocupacionales frente a la percepción del costo) reagrupando en dos bloques con tamaño suficiente:

- **Profesionales asalariados:** Profesionista (62) + Técnico (55) = 117
- **Independientes / empleadores:** Patrón (3) + Trabajador por cuenta propia (72) = 75

La agrupación es conceptualmente coherente: reúne a quienes trabajan de forma subordinada y con perfil profesional, frente a quienes trabajan por cuenta propia o son dueños de su actividad. El Q2 se reformula como: **¿difiere la percepción del costo del transporte público entre profesionales asalariados y trabajadores independientes?** Con 117 vs. 75 casos, el contraste es estimable y mantiene el sentido de la pregunta inicial.

## 5. Derivar el modo principal (preview)

`modo_principal` se obtiene combinando las 22 columnas de `p1a_*` y quedándose, por persona, con el medio que usa con mayor frecuencia. También se derivan `metro_principal` y `usuario_tp`. Las etiquetas de cada modo se leen del propio diccionario para no depender del orden de las columnas.

In [4]:
p1a_cols = sorted(c for c in df.columns if re.fullmatch(r"p1a_\d+", c))

def grupo_modo(lbl):
    l = lbl.lower()
    if re.search(r"metro|tren urbano|tren ligero|suburbano", l): return "metro"
    if re.search(r"\btren\b|brt|metrob|eléctric|electric|trolebus|tranv", l): return "tp_masivo"
    if re.search(r"camión|camion|microb|colectivo|combi|autobús|autobus", l): return "tp_concesionado"
    if re.search(r"taxi|bicitaxi|mototaxi", l): return "taxi"
    if re.search(r"automóvil|automovil|moto", l): return "auto_moto"
    if re.search(r"bicicleta|patín|patin|caminar|pie", l): return "activo"
    return "otro"

MODO_GRUPO = {c: grupo_modo(etiqueta(c)) for c in p1a_cols}
print("Modos detectados (columna -> grupo):")
for c in p1a_cols:
    print(f"  {c:9} {grupo_modo(etiqueta(c)):16} {etiqueta(c)[:60]}")

# Matriz de frecuencias; el modo principal es el de código mínimo por fila
frec = df[p1a_cols]
idx_min = frec.idxmin(axis=1)                  # columna con la frecuencia más alta
freq_min = frec.min(axis=1)
modo_principal = idx_min.map(MODO_GRUPO)
modo_principal[freq_min.isna() | (freq_min >= 3)] = np.nan  # nadie con uso cotidiano/ocasional

df["modo_principal"] = modo_principal
df["metro_principal"] = np.where(modo_principal.isna(), np.nan,
                                 (modo_principal == "metro").astype("Int64"))
tp = {"metro", "tp_masivo", "tp_concesionado"}
df["usuario_tp"] = modo_principal.isin(tp).astype("Int64")

print("\nDistribución de modo_principal:")
display(df["modo_principal"].value_counts(dropna=False).to_frame("n"))

Modos detectados (columna -> grupo):
  p1a_1     tp_masivo        1a  ¿Con que frecuencia utilizas los siguientes medios de tr
  p1a_10    otro             1a  ¿Con que frecuencia utilizas los siguientes medios de tr
  p1a_11    otro             1a  ¿Con que frecuencia utilizas los siguientes medios de tr
  p1a_12    auto_moto        1a  ¿Con que frecuencia utilizas los siguientes medios de tr
  p1a_13    otro             1a  ¿Con que frecuencia utilizas los siguientes medios de tr
  p1a_14    otro             1a  ¿Con que frecuencia utilizas los siguientes medios de tr
  p1a_15    auto_moto        1a  ¿Con que frecuencia utilizas los siguientes medios de tr
  p1a_16    activo           1a  ¿Con que frecuencia utilizas los siguientes medios de tr
  p1a_17    activo           1a  ¿Con que frecuencia utilizas los siguientes medios de tr
  p1a_18    otro             1a  ¿Con que frecuencia utilizas los siguientes medios de tr
  p1a_19    otro             1a  ¿Con que frecuencia utilizas l

,n
modo_principal,
tp_concesionado,579
auto_moto,320
activo,77
taxi,67
metro,65
otro,62
tp_masivo,15
NaN,6


## 6. Construir y exportar el subconjunto para la red

Reúne las columnas confirmadas más las derivadas y las guarda en `enmt_bn.csv`, la base reducida con la que se modelará. Además exporta `mapa_nodos.csv`, que documenta de dónde salió cada nodo y su cobertura, para dejar trazabilidad del proceso.

In [5]:
# Columnas crudas confirmadas (las presentes) + derivadas
cols_ok = [r["col"] for _, r in reporte.iterrows() if r["estado"] == "ok"]
derivadas = ["modo_principal", "metro_principal", "usuario_tp"]
cols_bn = list(dict.fromkeys(cols_ok + derivadas))   # sin duplicados, en orden

df_bn = df[cols_bn].copy()

RUTA_BN = DIR_PROC / "enmt_bn.csv"
RUTA_MAPA = DIR_PROC / "mapa_nodos.csv"
df_bn.to_csv(RUTA_BN, index=False, encoding="utf-8")
reporte.to_csv(RUTA_MAPA, index=False, encoding="utf-8")

print(f"✓ {RUTA_BN.name}: {df_bn.shape[0]:,} x {df_bn.shape[1]:,}")
print(f"✓ {RUTA_MAPA.name}: mapa nodo→columna con cobertura")
print(f"\nColumnas incluidas ({len(cols_bn)}):\n  {cols_bn}")

✓ enmt_bn.csv: 1,191 x 77


✓ mapa_nodos.csv: mapa nodo→columna con cobertura

Columnas incluidas (77):
  ['region', 'tam_loc', 'sexo', 'edad_1', 'pondi2', 'escol', 'ing_ind', 'ing_fam', 'cond_act', 'h21_1', 'h21_2', 'h21_3', 'h21_4', 'h21_5', 'h21_6', 'p1a_1', 'p1a_10', 'p1a_11', 'p1a_12', 'p1a_13', 'p1a_14', 'p1a_15', 'p1a_16', 'p1a_17', 'p1a_18', 'p1a_19', 'p1a_2', 'p1a_20', 'p1a_21', 'p1a_22', 'p1a_3', 'p1a_4', 'p1a_5', 'p1a_6', 'p1a_7', 'p1a_8', 'p1a_9', 'p17_4', 'p17_1', 'p17_2', 'p17_3', 'p1c_10_2', 'p1c_11_2', 'p1c_12_2', 'p1c_13_2', 'p1c_14_2', 'p1c_15_2', 'p1c_16_2', 'p1c_17_2', 'p1c_18_2', 'p1c_19_2', 'p1c_1_2', 'p1c_21_2', 'p1c_22_2', 'p1c_2_2', 'p1c_3_2', 'p1c_4_2', 'p1c_5_2', 'p1c_6_2', 'p1c_7_2', 'p1c_8_2', 'p1c_9_2', 'p1c_10_6', 'p1c_11_6', 'p1c_1_6', 'p1c_2_6', 'p1c_3_6', 'p1c_4_6', 'p1c_5_6', 'p1c_6_6', 'p1c_7_6', 'p1c_8_6', 'p1c_9_6', 'p25_1_2', 'modo_principal', 'metro_principal', 'usuario_tp']


## 7. Conclusiones de la exploración

Este notebook confirmó, contra la base ya limpia, que las variables necesarias para los cuatro queries existen y evaluó su calidad. Hallazgos principales:

- **Cobertura sólida en los nodos centrales.** Las variables de contexto (`region`, `tam_loc`, `sexo`, `edad_1`), las percepciones del transporte público (`p17_1` a `p17_4`), la escolaridad (`escol`), el ingreso (`ing_ind`) y el asalto en TP (`p25_1_2`) tienen ~0% de faltantes. La batería de uso de modos (`p1a_*`) también.

- **La percepción por modo no es usable.** Las baterías `p1c_*` (calificación de seguridad y costo por cada medio) tienen 90-93% de faltantes, porque cada persona solo calificó los modos que usa. Se descartan. En su lugar se usan las variables `p17_*`, que miden la percepción del transporte público en general y están completas.

- **La ocupación se recuperó a nivel individuo.** No existe una variable directa de ocupación del informante; se reconstruyó enlazando el roster (`h21_*`) con `h8` (individuo seleccionado). Se recuperó ocupación para 700 informantes. El resto son faltantes esperados (estudiantes, jubilados, personas sin actividad económica).

- **`modo_principal` es una variable derivada, no una columna.** Se construyó a partir de `p1a_*` tomando el modo de mayor frecuencia de uso por persona; de ahí salen también `metro_principal` y `usuario_tp`.

- **Escolaridad e ingreso ya vienen categorizados.** `escol` tiene 4 niveles (Primaria, Secundaria, Preparatoria, Universidad/Posgrado) e `ing_ind` 4 tramos en salarios mínimos. No requieren discretización, solo posibles reagrupaciones.

- **Q4 es viable pero con evento raro.** Solo 74 de 1,191 personas reportaron asalto en TP. La pregunta se sostiene, pero al cruzar por ingreso y restringir a usuarios de TP las celdas quedan pequeñas; los resultados deben interpretarse con cautela.

El producto de esta etapa es `enmt_bn.csv` (subconjunto de variables en crudo)  `mapa_nodos.csv` (trazabilidad de cada nodo a su columna de origen y su cobertura).

## 8. Siguientes pasos (notebook 03)

Este notebook deja las variables **identificadas y verificadas, pero en crudo**. El notebook 03 debe dejarlas en su forma final para el análisis. 

### Pendientes:

**Recodificaciones por nodo**
- `ocupacion_ind`: crear la versión agrupada de dos categorías para el análisis. Conservar también la versión detallada de 14 categorías por si se requiere describirla.
- `ing_ind` → variable binaria "ingreso bajo / alto", partiendo en la **mediana de los usuarios de TP**, no de toda la muestra (Q4 se plantea entre usuarios de TP).
- `efectividad` (Q1): construir un nodo único a partir de `p17_1` (eficiente) y `p17_2` (rápido). La forma sugerida es booleana (efectivo = eficiente + rápido) o una escala ordinal de tres niveles; queda a criterio de quien modele.
- `escol`: usable con sus 4 niveles; si alguna celda queda escasa, colapsar a básica / media superior / superior.

**Chequeos rápidos pendientes**
- El reporte contó 5 valores únicos en `escol` e `ing_ind`, pero el diccionario solo documenta 4 etiquetas en cada uno. Verificar qué es ese quinto valor (posible "sin escolaridad" / "sin ingreso" sin etiqueta) antes de recodificar.
- Contar cuántos de los 74 asaltos ocurren **dentro** de los usuarios de TP, para dimensionar con precisión la celda de Q4.

**Tratamiento de faltantes y universos**
- Decidir, por nodo, si los NaN se filtran o se tratan como categoría propia.
- Aplicar los filtros de universo: Q1 y Q4 se plantean solo entre usuarios de TP (`usuario_tp`).
